# 03 — Feature Engineering

Tạo feature tại đúng ngày dự báo `t`. Không dùng giao dịch từ `t` trở đi. File đầu ra chỉ giữ một phiên bản của Frequency/Monetary/TotalItems; `Customer ID` là khóa để nối nhãn, **không phải đầu vào mô hình**.

In [1]:
import pandas as pd
import numpy as np
import os

input_path = '../data/interim/customer_base_model.csv'
output_path = '../data/processed/customer_features.csv'

prediction_date = pd.Timestamp('2011-06-01')
df = pd.read_csv(input_path)
df['Last_Purchase_Date'] = pd.to_datetime(df['Last_Purchase_Date'], errors='coerce')
print('Số khách đầu vào:', len(df))

Số khách đầu vào: 2658


In [2]:
# RFM + đặc trưng bổ sung tại t
df['Recency'] = (prediction_date - df['Last_Purchase_Date'].dt.normalize()).dt.days
df['AvgOrderValue'] = np.where(df['Frequency'] > 0, df['Monetary'] / df['Frequency'], 0)

# Không đưa các cột ngày và cột trùng vào tập feature cuối.
feature_cols = [
    'Customer ID', 'Recency', 'Frequency', 'Monetary',
    'AvgOrderValue', 'UniqueProducts', 'TotalItems', 'Country'
]
features = df[feature_cols].copy()
features = features.replace([np.inf, -np.inf], np.nan)
features['AvgOrderValue'] = features['AvgOrderValue'].fillna(0)

# Kiểm tra khách nhiều quốc gia sẽ được xử lý ở bước mô hình bằng Country đã chốt tại khoảng lịch sử.
print(features.head().T)
print('\nMissing values:')
print(features.isna().sum())
print('\nThống kê RFM:')
print(features[['Recency','Frequency','Monetary']].describe())

                             0        1        2        3           4
Customer ID            12346.0  12347.0  12348.0  12350.0     12352.0
Recency                    134       55       57      119          71
Frequency                    2        4        4        1           6
Monetary                137.81  2434.96   1388.4    294.4      864.98
AvgOrderValue           68.905   608.74    347.1    294.4  144.163333
UniqueProducts              20       90       24       16          36
TotalItems                  18     1626     2488      196         374
Country         United Kingdom  Iceland  Finland   Norway      Norway

Missing values:
Customer ID       0
Recency           0
Frequency         0
Monetary          0
AvgOrderValue     0
UniqueProducts    0
TotalItems        0
Country           0
dtype: int64

Thống kê RFM:
           Recency    Frequency       Monetary
count  2658.000000  2658.000000    2658.000000
mean     61.440933     5.588036    2536.068282
std      49.967137     8

In [3]:
os.makedirs('../data/processed', exist_ok=True)
features.to_csv(output_path, index=False)
print(f'Đã lưu {output_path} | {len(features)} khách')
print('Lưu ý: Customer ID chỉ dùng để nối nhãn, sẽ loại khỏi X khi train model.')

Đã lưu ../data/processed/customer_features.csv | 2658 khách
Lưu ý: Customer ID chỉ dùng để nối nhãn, sẽ loại khỏi X khi train model.
